In [1]:
import pandas as pd
from lightgbm import LGBMClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import StandardScaler
from utils import *
import numpy as np
import wandb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.model_selection import TimeSeriesSplit


In [2]:
df = pd.read_csv('../../../data/creditcard.csv')
df = create_features(df)
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V25,V26,V27,V28,Amount,Class,_log_amount,Hour_from_start_mod24,is_night_proxy,is_business_hours_proxy
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0.128539,-0.189115,0.133558,-0.021053,149.62,0,5.014760,0,1,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,0.167170,0.125895,-0.008983,0.014724,2.69,0,1.305626,0,1,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0,5.939276,0,1,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,0.647376,-0.221929,0.062723,0.061458,123.50,0,4.824306,0,1,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.206010,0.502292,0.219422,0.215153,69.99,0,4.262539,0,1,0


In [3]:
features = [c for c in df.columns if c.startswith("V")] + [
    'Amount','_log_amount',
    'Hour_from_start_mod24','is_night_proxy','is_business_hours_proxy'
]
target = "Class"

In [4]:
X_train, y_train, X_val, y_val, X_test, y_test = split_data(df, features, target)

X_train: (181584, 33) y_train: (181584,)
X_val: (45396, 33) y_val: (45396,)
X_test: (56746, 33) y_test: (56746,)
Fraud rate in train: 0.001910961318177813
Fraud rate in test: 0.0013040566735981391


### Logistic Regression

In [5]:
logit_pipe = ImbPipeline(steps=[
    ("scaler", StandardScaler(with_mean=True)),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))
])
logit_pipe.fit(X_train, y_train)

val_log_proba  = logit_pipe.predict_proba(X_val)[:, 1]
test_log_proba = logit_pipe.predict_proba(X_test)[:, 1]

In [6]:
result_lr_val = log_eval(y_val, val_log_proba)
result_lr_test = log_eval(y_test, test_log_proba)
print("Logistic Regression - Validation Set:")
print(result_lr_val)
print("Logistic Regression - Test Set:")
print(result_lr_test)

Logistic Regression - Validation Set:
{'threshold': 0.907, 'Cost': 2400.0, 'ROC_AUC': 0.9729554584441671, 'PR_AUC': 0.7749693650102897}
Logistic Regression - Test Set:
{'threshold': 0.969, 'Cost': 2980.0, 'ROC_AUC': 0.9856811171349215, 'PR_AUC': 0.7414676125025982}


### Random Forest

In [7]:
rf_pipe = ImbPipeline(steps=[
    ("clf", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1
    ))
])
rf_pipe.fit(X_train, y_train)

val_rf_proba  = rf_pipe.predict_proba(X_val)[:, 1]
test_rf_proba = rf_pipe.predict_proba(X_test)[:, 1]


In [8]:
result_rf_val = log_eval(y_val, val_rf_proba)
result_rf_test = log_eval(y_test, test_rf_proba)
print("Random Forest - Validation Set:")
print(result_rf_val)
print("Random Forest - Test Set:")
print(result_rf_test)

Random Forest - Validation Set:
{'threshold': 0.027, 'Cost': 2250.0, 'ROC_AUC': 0.9367508125916075, 'PR_AUC': 0.7686420745950832}
Random Forest - Test Set:
{'threshold': 0.031, 'Cost': 3060.0, 'ROC_AUC': 0.9540934700581439, 'PR_AUC': 0.8053069823934353}


### XGBoost

In [9]:
pos, neg = int((y_train==1).sum()), int((y_train==0).sum())
spw = neg / max(1, pos)
xgb_pipe = ImbPipeline(steps=[
    ("model", XGBClassifier(
        n_estimators=600, max_depth=4, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
        tree_method="hist", random_state=SEED,
        scale_pos_weight=spw, n_jobs=-1, eval_metric="aucpr"
    ))
])
xgb_pipe.fit(X_train, y_train)

val_xgb_proba  = xgb_pipe.predict_proba(X_val)[:, 1]
test_xgb_proba = xgb_pipe.predict_proba(X_test)[:, 1]

In [10]:
result_xgb_val = log_eval(y_val, val_xgb_proba)
result_xgb_test = log_eval(y_test, test_xgb_proba)
print("XGBoost - Validation Set:")
print(result_xgb_val)
print("XGBoost - Test Set:")
print(result_xgb_test)

XGBoost - Validation Set:
{'threshold': 0.08, 'Cost': 2405.0, 'ROC_AUC': 0.9824330078443082, 'PR_AUC': 0.7721461819041088}
XGBoost - Test Set:
{'threshold': 0.105, 'Cost': 2965.0, 'ROC_AUC': 0.9769317418773941, 'PR_AUC': 0.7946662673961108}


### Ensemble XGB + RF

In [11]:
final_val_proba = 0.6 * val_xgb_proba + 0.4 * val_rf_proba

best_th, _ = thr_min_cost(y_val, final_val_proba)

final_test_proba = 0.6 * test_xgb_proba + 0.4 * test_rf_proba
y_test_pred = (final_test_proba >= best_th).astype(int)
print("Ensemble (XGB + RF) - Test Set:")
print(log_eval(y_test, final_test_proba))
print("Ensemble (XGB + RF) - Val Set:")
print(log_eval(y_val, final_val_proba))
print("Best Threshold:", best_th)
ece_xgb_rf_test =adaptive_ece(y_test, final_test_proba, n_bins=15)
ece_xgb_rf_val = adaptive_ece(y_val, final_val_proba, n_bins=15)
print(f"Expected Calibration Error - Test (Ensemble): {ece_xgb_rf_test:.4f}")
print(f"Expected Calibration Error - Val (Ensemble): {ece_xgb_rf_val:.4f}")

Ensemble (XGB + RF) - Test Set:
{'threshold': 0.08600000000000001, 'Cost': 2930.0, 'ROC_AUC': 0.9807109569337831, 'PR_AUC': 0.808047758193593}
Ensemble (XGB + RF) - Val Set:
{'threshold': 0.074, 'Cost': 2375.0, 'ROC_AUC': 0.9787555643016123, 'PR_AUC': 0.7664699042405145}
Best Threshold: 0.074
Expected Calibration Error - Test (Ensemble): 0.0003
Expected Calibration Error - Val (Ensemble): 0.0003


### LightGBM

In [12]:
lgb_pipe = ImbPipeline(steps=[
    ("model", LGBMClassifier(
        n_estimators=600,
        learning_rate=0.05,
        num_leaves=20,
        max_depth=4,
        min_child_samples=50,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        class_weight="balanced",
        objective="binary",
        random_state=SEED,
        n_jobs=-1
    ))
])
lgb_pipe.fit(X_train, y_train)

val_lgb_proba  = lgb_pipe.predict_proba(X_val)[:, 1]
test_lgb_proba = lgb_pipe.predict_proba(X_test)[:, 1]

[LightGBM] [Info] Number of positive: 347, number of negative: 181237
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.020342 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7678
[LightGBM] [Info] Number of data points in the train set: 181584, number of used features: 33
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gai

In [13]:
result_lgb_val = log_eval(y_val, val_lgb_proba)
result_lgb_test = log_eval(y_test, test_lgb_proba)
print("LightGBM - Validation Set:")
print(result_lgb_val)
print("LightGBM - Test Set:")
print(result_lgb_test)

LightGBM - Validation Set:
{'threshold': 0.062, 'Cost': 2525.0, 'ROC_AUC': 0.9778755394658271, 'PR_AUC': 0.7740247965241721}
LightGBM - Test Set:
{'threshold': 0.222, 'Cost': 3115.0, 'ROC_AUC': 0.9739391777435256, 'PR_AUC': 0.7937323689038129}


## Calibration (Isotonic) with TimeSeriesSplit on TRAIN

In [14]:
cal_cv = TimeSeriesSplit(n_splits=3)

### Calibrated XGBoost

In [15]:
xgb_cal = CalibratedClassifierCV(estimator=xgb_pipe, method="isotonic", cv=cal_cv)
xgb_cal.fit(X_train, y_train)
val_xgb_cal  = xgb_cal.predict_proba(X_val)[:, 1]
test_xgb_cal = xgb_cal.predict_proba(X_test)[:, 1]

In [16]:
result_xgb_cal_val = log_eval(y_val, val_xgb_cal)
result_xgb_cal_test = log_eval(y_test, test_xgb_cal)
print("Calibrated XGBoost - Validation Set:")
print(result_xgb_cal_val)
print("Calibrated XGBoost - Test Set:")
print(result_xgb_cal_test)
print("ECE - XGBoost - Validation Set:", expected_calibration_error(y_val, val_xgb_cal, n_bins=15))
print("ECE - XGBoost - Test Set:", expected_calibration_error(y_test, test_xgb_cal, n_bins=15))

Calibrated XGBoost - Validation Set:
{'threshold': 0.189, 'Cost': 2230.0, 'ROC_AUC': 0.9786809212312035, 'PR_AUC': 0.7652545184568359}
Calibrated XGBoost - Test Set:
{'threshold': 0.113, 'Cost': 3065.0, 'ROC_AUC': 0.9703460739466174, 'PR_AUC': 0.759713062104826}
ECE - XGBoost - Validation Set: 0.0008425815042460422
ECE - XGBoost - Test Set: 0.0006412869134827822


### Calibrated Random Forest

In [17]:
rf_cal = CalibratedClassifierCV(estimator=rf_pipe, method="isotonic", cv=cal_cv)
rf_cal.fit(X_train, y_train)
val_rf_cal  = rf_cal.predict_proba(X_val)[:, 1]
test_rf_cal = rf_cal.predict_proba(X_test)[:, 1]

In [19]:
result_rf_cal_val = log_eval(y_val, val_rf_cal)
result_rf_cal_test = log_eval(y_test, test_rf_cal)
print("Calibrated Random Forest - Validation Set:")
print(result_rf_cal_val)
print("Calibrated Random Forest - Test Set:")
print(result_rf_cal_test)
print("ECE - Random Forest - Validation Set:", adaptive_ece(y_val, val_rf_cal, n_bins=15))
print("ECE - Random Forest - Test Set:", adaptive_ece(y_test, test_rf_cal, n_bins=15))

Calibrated Random Forest - Validation Set:
{'threshold': 0.065, 'Cost': 2215.0, 'ROC_AUC': 0.9730956262553607, 'PR_AUC': 0.7500814806131584}
Calibrated Random Forest - Test Set:
{'threshold': 0.009000000000000001, 'Cost': 3045.0, 'ROC_AUC': 0.9548839362018711, 'PR_AUC': 0.7893729914516632}
ECE - Random Forest - Validation Set: 0.0004330446136923579
ECE - Random Forest - Test Set: 0.0003597413319785475
